# 🏥 Transparent AI Framework for Healthcare Claim Denial Explanations
## Sprint 2: Data Preprocessing, Leakage Mitigation & Baseline Modeling
**Master of Data Science — University of Europe for Applied Sciences**

**Team:** Dennis Sebastian (59593398) & Saran Jayakumaran Nair Sasikala (56500625)  
**Supervisor:** Dr. Humera Noor  
**Reporting Period:** 09 Nov 2026 – 22 Nov 2026 | **Submission Date:** 23 Nov 2026

---

### 🎯 Sprint 2 Objectives & Scope
1. Build a reproducible preprocessing pipeline for the California DMHC IMR dataset (42,749 raw records).
2. Audit data hygiene: remove 39 exact duplicate rows and resolve 1,378 rows with missing demographic/review values.
3. Mitigate data leakage: examine post-determination artifacts (`DaysToAdopt`) and text-level label exposure (`Findings`).
4. Encode features: ordinal encoding for `AgeRange` (0–6), label encoding for high-cardinality clinical categories.
5. Perform stratified 80/20 train/test split to preserve natural target class balance (~52.9% Overturned vs 47.1% Upheld).
6. Train and evaluate 3 benchmark baseline models: Logistic Regression, Random Forest, and Hist Gradient Boosting.
7. Save model-ready datasets and transformation artifacts for downstream XAI (SHAP/LIME) in Sprints 3–5.

## 1. Imports and Environment Setup

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150
print('Libraries and plotting environment loaded successfully.')

## 2. Ingestion & Data Cleansing Pipeline

In [2]:
raw_path = 'independent-medical-review-imr-determinations-trends.csv'
df_raw = pd.read_csv(raw_path)
df_raw.columns = df_raw.columns.str.strip()
print('Raw Ingestion Shape:', df_raw.shape)

# Step 2.1: Deduplication
n_dups = df_raw.duplicated().sum()
print('Duplicate Rows Flagged & Removed:', n_dups)
df_dedup = df_raw.drop_duplicates().copy()

# Step 2.2: Complete Case Handling for Key Nulls
crit_cols = ['AgeRange', 'PatientGender', 'DaysToReview', 'DiagnosisSubCategory', 'TreatmentSubCategory']
print('Missing counts in deduplicated subset:')
print(df_dedup[crit_cols].isnull().sum())

df_clean = df_dedup.dropna(subset=crit_cols).copy()
print('Cleaned Complete Case Shape:', df_clean.shape)
print(f'Retention Rate: {len(df_clean)/len(df_raw)*100:.2f}%')

## 3. Data Leakage Assessment & Target Encoding
In healthcare dispute analytics, post-decision variables can leak the label:
- `ReferenceID`: Administrative tracker, dropped.
- `Findings`: Contains explicit conclusion phrases (e.g., 'Therefore, the Health Plan denial should be upheld'). Excluded from baseline tabular modeling to prevent direct text target leakage.
- `DaysToAdopt`: Measures administrative closure duration; retained for timing evaluation but monitored closely.
- Target `Determination`: Mapped to binary integer (`Overturned Decision of Health Plan` = 1, `Upheld Decision of Health Plan` = 0).

In [3]:
df_clean['Target'] = (df_clean['Determination'] == 'Overturned Decision of Health Plan').astype(int)
print('Target Class Distribution:')
print(df_clean['Target'].value_counts(normalize=True).rename({1: 'Overturned (1)', 0: 'Upheld (0)'}))

## 4. Feature Engineering: Ordinal & Categorical Encodings

In [4]:
# Ordinal Age Mapping
age_map = {
    '0 to 10': 0, '11 to 20': 1, '21 to 30': 2, '31 to 40': 3,
    '41 to 50': 4, '51 to 64': 5, '65+': 6
}
df_clean['AgeRange_Ord'] = df_clean['AgeRange'].map(age_map)

# Categorical Encodings
cat_cols = ['PatientGender', 'Type', 'IMRType', 'DiagnosisCategory', 'DiagnosisSubCategory', 'TreatmentCategory', 'TreatmentSubCategory']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_clean[col + '_Enc'] = le.fit_transform(df_clean[col].astype(str))
    encoders[col] = le
encoders['AgeRange_Map'] = age_map

feature_cols = ['ReportYear', 'AgeRange_Ord', 'DaysToReview', 'DaysToAdopt'] + [c + '_Enc' for c in cat_cols]
X = df_clean[feature_cols]
y = df_clean['Target']
print('Engineered Feature Matrix:', X.shape)

## 5. Stratified 80/20 Train/Test Partition

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train records: {len(X_train):,}, Test records: {len(X_test):,}')
print(f'Train class ratio (Overturned): {y_train.mean():.4f}, Test class ratio: {y_test.mean():.4f}')

## 6. Baseline Model Training & Benchmarking

In [6]:
# Standard scaling for linear baseline
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=4)
rf.fit(X_train, y_train)

# 3. Hist Gradient Boosting (LightGBM equivalent)
hgb = HistGradientBoostingClassifier(max_iter=100, random_state=42)
hgb.fit(X_train, y_train)

models = {
    'Logistic Regression': (lr, X_test_scaled),
    'Random Forest': (rf, X_test),
    'Hist Gradient Boosting': (hgb, X_test)
}

results = []
for name, (mod, X_eval) in models.items():
    preds = mod.predict(X_eval)
    probs = mod.predict_proba(X_eval)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, probs)
    })
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

## 7. Model Evaluation & ROC Visualization

In [7]:
plt.figure(figsize=(7, 5))
colors = {'Logistic Regression': '#ef4444', 'Random Forest': '#3b82f6', 'Hist Gradient Boosting': '#10b981'}
for name, (mod, X_eval) in models.items():
    probs = mod.predict_proba(X_eval)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=colors[name], lw=2)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Baseline (AUC = 0.500)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Sprint 2 Baseline ROC-AUC Curves')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 8. Artifact Serialization for Sprint 3 & XAI
Persisting cleaned dataset, transformation encoders, scalers, and baseline models.

In [8]:
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)
df_clean.to_csv('data/processed/imr_processed.csv', index=False)
joblib.dump(encoders, 'models/encoders.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump({'lr': lr, 'rf': rf, 'hgb': hgb}, 'models/baseline_models.pkl')
print('All Sprint 2 preprocessing and baseline model artifacts persisted.')